# CS781 Phase 2 - L-NER - ENSEMBLE v3 (all adaptation techniques, CPU only)
Attach: ONE dataset holding **all** bundle zips (`bundle_a..e.zip` from the FINAL notebooks and `bundle_a_tech.zip .. bundle_e_tech.zip` from the technique notebooks),
the code dataset and `cs781-legal-data`. Internet ON, secret `HF_TOKEN`, **Accelerator: None**. No model weights needed.

Members: every run that has a dev twin and an all-data twin - techniques `full, lorma, lora, adapter, partial, lorma32, fullreg` on models B-E (model A and the norm-only runs are weak and left out of the ensembles, but listed as singles).
Averages per-token probabilities, BIO-Viterbi decode. For each member set two files:
* `ens_dev_<set>`   - the 49-document dev-stage models (best epoch) -> the only ones with a dev score
* `ens_final_<set>` - the same members trained on all 70 documents -> **the files to submit**
Sets: `core8` (B-E full+lorma = the ensemble already submitted), `techall` (all 28 runs), `peft` (LoRMA/LoRA/adapter variants only), `top10` (best 10 by dev), `top2permodel`, `bestpermodel`.


In [ ]:
import os, sys, json, re, glob, shutil, subprocess, time, threading
WORK = "/kaggle/working/p2code"
P2DATA = "/kaggle/working/p2data"
RUNS = "/kaggle/working/runs"

def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("ner_train.py"), WORK)
os.chdir(WORK)
TOK_ROOT = find_dir("manifest.json")
print("code:", WORK, sorted(f for f in os.listdir(WORK) if f.endswith(".py")))
print("tokenizer root:", TOK_ROOT, os.listdir(TOK_ROOT))

# ---- Phase-2 data from Hugging Face (Train folds + labels; Test fold3 is read only by ner_train.py for prediction)
os.makedirs(P2DATA, exist_ok=True)
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
for f in ["Train/fold1-text.json", "Train/fold1-gold.json", "Train/fold2-text.json", "Train/fold2-gold.json",
          "Train/labels.txt", "Test/fold3-text.json"]:
    p = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_2_LNER/{f}", repo_type="dataset", token=tok_)
    shutil.copy(p, os.path.join(P2DATA, os.path.basename(f)))
TEST = os.path.join(P2DATA, "fold3-text.json")
print("p2 data:", os.listdir(P2DATA), "test docs:", len(json.load(open(TEST))))

# ---- Phase-1 checkpoints: final.pt per model letter (continual-pretrained *_cpt preferred)
CKPT = {}
for p in sorted(glob.glob("/kaggle/input/**/final.pt", recursive=True)):
    m = re.search(r"run_([a-e])_[^/]*", p)
    if not m or "smoke" in p:
        continue
    k = m.group(1)
    if k not in CKPT or ("_cpt" in p and "_cpt" not in CKPT[k]):
        CKPT[k] = p
for k, v in sorted(CKPT.items()):
    print(f"  model {k}: {v}")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
os.makedirs("/kaggle/working/logs", exist_ok=True)

def run_parallel(jobs, timeout=None):
    """jobs: list of (tag, shell_cmd, gpu_index_or_None). Runs them at the same time (one per GPU), streams
    output prefixed with the tag, saves /kaggle/working/logs/<tag>.txt, returns {tag: (exit_code, text)}."""
    procs, outs, lock = {}, {}, threading.Lock()
    def reader(tag, p):
        buf = []
        for line in p.stdout:
            buf.append(line)
            with lock:
                print(f"[{tag}] {line}", end="", flush=True)
        outs[tag] = buf
    threads = []
    for tag, cmd, gpu in jobs:
        env = dict(os.environ)
        if gpu is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        print(f"$ [{tag}] (GPU {gpu}) {cmd}", flush=True)
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env=env, start_new_session=True)
        procs[tag] = p
        t = threading.Thread(target=reader, args=(tag, p), daemon=True); t.start(); threads.append(t)
    timer = None
    if timeout:
        def kill():
            print("TIMEOUT - killing", flush=True)
            for p in procs.values():
                try: os.killpg(p.pid, 9)
                except Exception: pass
        timer = threading.Timer(timeout, kill); timer.start()
    for p in procs.values(): p.wait()
    for t in threads: t.join()
    if timer: timer.cancel()
    res = {}
    for tag, p in procs.items():
        txt = "".join(outs.get(tag, []))
        open(f"/kaggle/working/logs/{tag}.txt", "w").write(txt)
        res[tag] = (p.returncode, txt)
        print(f"[{tag}] exit {p.returncode}", flush=True)
    return res

## 1. Collect runs

In [ ]:
# ---- collect the run folders of all model notebooks (bundle zips and/or notebook outputs attached as inputs)
import zipfile
os.makedirs(RUNS, exist_ok=True)
for z in glob.glob("/kaggle/input/**/bundle_*.zip", recursive=True):
    zipfile.ZipFile(z).extractall("/kaggle/working/_bundles")
FOUND = {}
for root in ("/kaggle/input", "/kaggle/working/_bundles"):
    for p in glob.glob(f"{root}/**/test_logp.npz", recursive=True):
        d = os.path.dirname(p)
        FOUND[os.path.basename(d)] = d
for tag, d in FOUND.items():
    if not os.path.exists(f"{RUNS}/{tag}"):
        shutil.copytree(d, f"{RUNS}/{tag}")
print("runs found:", sorted(FOUND))

## 2. Ensembles

In [ ]:
# ---- ensembles: average of per-token probabilities, BIO-Viterbi decode
TECHS = ["full", "lorma", "lora", "adapter", "partial", "lorma32", "fullreg", "norm"]
def have(tag, dev=True):
    return all(os.path.exists(f"{RUNS}/{tag}/{f}") for f in (("summary.json", "test_logp.npz") + (("dev_logp.npz",) if dev else ())))
def dev_f1(tag):
    return json.load(open(f"{RUNS}/{tag}/summary.json"))["dev_viterbi"]["macro_f1"]
DEV_TAGS = [f"{m}_{t}" for m in "abcde" for t in TECHS if have(f"{m}_{t}")]
POOL = [t for t in DEV_TAGS if not t.startswith("a_") and not t.endswith("_norm")]        # B-E, no norm-only
ranked = sorted(POOL, key=dev_f1, reverse=True)
print("dev ranking (pool):", [(t, round(dev_f1(t), 4)) for t in ranked])
ENS, MEMBERS = [], {}
def run_ens(name, tags):
    out = f"{RUNS}/{name}"
    t0 = time.time()
    r = subprocess.run(f"python ner_ensemble.py --runs {' '.join(f'{RUNS}/{t}' for t in tags)} --tok_root {TOK_ROOT} --data_dir {P2DATA} "
                       f"--test_file {TEST} --out {out}", shell=True, capture_output=True, text=True, cwd=WORK)
    print(f"=== {name} ({len(tags)} members, {time.time() - t0:.0f}s)\n", r.stdout[-300:], r.stderr[-1200:])
    if r.returncode == 0:
        ENS.append(name); MEMBERS[name] = tags
def per_model(k):
    out = []
    for m in "bcde":
        c = sorted([t for t in POOL if t.startswith(m + "_")], key=dev_f1, reverse=True)
        out += c[:k]
    return out
SETS = {"core8": [t for t in POOL if t.split("_", 1)[1] in ("full", "lorma")],
        "techall": POOL,
        "peft": [t for t in POOL if t.split("_", 1)[1] in ("lorma", "lora", "adapter", "lorma32")],
        "top10": ranked[:10],
        "top2permodel": per_model(2),
        "bestpermodel": per_model(1)}
done = set()
for nm, tags in SETS.items():
    if len(tags) < 2 or tuple(sorted(tags)) in done:
        continue
    done.add(tuple(sorted(tags)))
    run_ens(f"ens_dev_{nm}", tags)                                            # measurable on the 21 dev docs
    ft = [t + "_all" for t in tags if have(t + "_all", dev=False)]
    if len(ft) >= 2:
        run_ens(f"ens_final_{nm}", ft)                                        # same members, all-data models -> submit these


## 3. Sanity check

In [ ]:
# ---- sanity check: compare each all-data model's TEST predictions with its dev-stage twin (span agreement F1; healthy ~0.7-0.9)
import zipfile
def load_pred(tag):
    p = f"{RUNS}/{tag}/prediction.json"
    if os.path.exists(p):
        return json.load(open(p))
    z = f"{RUNS}/{tag}/submission.zip"
    if os.path.exists(z):
        with zipfile.ZipFile(z) as zz:
            return json.loads(zz.read("prediction.json"))
    return None
def spanset(p):
    return {(d, int(s[0]), int(s[1]), s[2]) for d, v in p.items() for s in v}
def agree(p, q):
    P, Q = spanset(p), spanset(q)
    return 2 * len(P & Q) / (len(P) + len(Q)) if (P or Q) else 1.0
CHK = ["SANITY (test predictions): all-data run | entities | agreement with its dev-stage twin"]
bad = []
for t in DEV_TAGS:
    a_, b_ = load_pred(t), load_pred(t + "_all")
    if a_ is not None and b_ is not None:
        ag = agree(a_, b_)
        CHK.append(f"{t + '_all':18s} | {len(spanset(b_)):5d} | {ag:.3f}" + ("   <-- CHECK" if ag < 0.6 else ""))
        if ag < 0.6: bad.append(t)
for t in ENS:
    if t.startswith("ens_final"):
        twin = "ens_dev_" + t.split("_", 2)[2]
        p, q = load_pred(t), load_pred(twin)
        if p is not None and q is not None:
            CHK.append(f"{t:26s} | {len(spanset(p)):5d} | agreement with {twin}: {agree(p, q):.3f}")
print("\n".join(CHK)); print("suspicious:", bad)


## 4. Report

In [ ]:
# ---- report + submissions
LABS = [l.strip() for l in open(f"{P2DATA}/labels.txt") if l.strip()]
os.makedirs("/kaggle/working/submissions", exist_ok=True)
L = ["SINGLE RUNS (dev macro-F1, 21 docs), sorted"]
per = {}
for t in sorted(DEV_TAGS, key=dev_f1, reverse=True):
    s = json.load(open(f"{RUNS}/{t}/summary.json")); dv = s["dev_viterbi"]
    per[t] = {l: dv["per_label"][l]["f1"] for l in LABS}
    L.append(f"{t:14s} | {dv['macro_f1']:.4f} | greedy {s['dev_greedy']['macro_f1']:.4f} | best ep {s['best_epoch']} | {s['params_trainable'] / 1e6:.2f}M trainable")
L.append("\ntechnique x model (dev macro-F1)")
L.append(f"{'':9s}" + "".join(f"{m.upper():>8s}" for m in "abcde") + f"{'mean B-E':>10s}")
for tc in TECHS:
    row, vals = [], []
    for m in "abcde":
        v = dev_f1(f"{m}_{tc}") if f"{m}_{tc}" in DEV_TAGS else None
        row.append(f"{v:8.4f}" if v is not None else f"{'-':>8s}")
        if v is not None and m != "a": vals.append(v)
    L.append(f"{tc:9s}" + "".join(row) + (f"{sum(vals) / len(vals):10.4f}" if vals else ""))
L.append("\nDEV ENSEMBLES (49-doc models; macro-F1 on the 21 dev docs)")
for t in ENS:
    if t.startswith("ens_dev"):
        sj = json.load(open(f"{RUNS}/{t}/summary.json")).get("dev", {})
        if sj:
            per[t] = {l: sj["per_label"][l]["f1"] for l in LABS}
            L.append(f"{t:22s} | {sj['macro_f1']:.4f} | {len(MEMBERS[t])} members")
L.append("\nFINAL ENSEMBLES (all-data models; no dev score)")
for t in ENS:
    if t.startswith("ens_final"):
        L.append(f"{t:22s} | {len(MEMBERS[t])} members")
L.append("\nper-label dev F1 (ensembles)")
L.append(f"{'run':22s} " + " ".join(f"{l[:7]:>7s}" for l in LABS))
for t, d in per.items():
    if t.startswith("ens_"):
        L.append(f"{t:22s} " + " ".join(f"{d[l]:7.2f}" for l in LABS))
L.append("")
L += CHK
for t in ENS:
    if os.path.exists(f"{RUNS}/{t}/submission.zip"):
        shutil.copy(f"{RUNS}/{t}/submission.zip", f"/kaggle/working/submissions/{t}.zip")
L.append("\nsubmission files: " + ", ".join(sorted(os.listdir("/kaggle/working/submissions"))))
rep = "\n".join(L)
open("/kaggle/working/REPORT.txt", "w").write(rep)
print("=== REPORT ===\n" + rep + "\n=== END REPORT ===")
